## Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DecimalType
from pyspark.sql.functions import col, lower, upper, count, when, trim, round, datediff, year, month, lit, to_date, to_timestamp

## Reading from bronze layer

In [0]:
df = spark.table("olist.bronze.order_items")
df.display()

## Overview about the table

In [0]:
# Table info
print("=== Schema ===")
df.printSchema()

print("=== Row Count ===")
print(f"Total rows: {df.count()}")

print("=== Null Counts per Column ===")
df.select([
    F.count(F.when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()



## Transformations

### 1. Trim all whitespaces from string columns

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


### 2. Normalize improperly represented nulls in string columns

In [0]:
NULL_STRINGS = ["", "null", "none", "n/a", "na", "unknown", "-"]

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(
            field.name,
            when(lower(trim(col(field.name))).isin(NULL_STRINGS), None)
            .otherwise(col(field.name)))
    

### 3. Cast monetary columns to DECIMAL(10,2)

In [0]:
# Keep cent-level precision for accurate gold-layer revenue calculations.
# DECIMAL(10,2) is the standard type for currency in data warehousing.
df = df.withColumn("price", col("price").cast(DecimalType(10, 2))) \
       .withColumn("freight_value", col("freight_value").cast(DecimalType(10, 2)))

### 4. Validate numeric values

In [0]:
# price should be > 0 (a free item doesn't make sense for a paid order_item)
# freight_value should be >= 0
df = df.withColumn(
    "has_valid_amounts",
    (col("price") > 0) & (col("freight_value") >= 0)
)

### 5. Derive useful analytical columns

In [0]:
# total_item_value: product price + freight (what the customer effectively paid for this line)
# freight_ratio: freight as a share of price — useful for margin analysis
df = df.withColumn(
    "total_item_value",
    (col("price") + col("freight_value")).cast(DecimalType(12, 2))
).withColumn(
    "freight_ratio",
    when(col("price") > 0, round(col("freight_value") / col("price"), 4))
    .otherwise(None)
)

### 6. Add partition-friendly date columns

In [0]:
df = df.withColumn("shipping_limit_date_only", to_date(col("shipping_limit_date"))) \
       .withColumn("shipping_limit_year", year(col("shipping_limit_date"))) \
       .withColumn("shipping_limit_month", month(col("shipping_limit_date")))

### 7. Handle nulls

In [0]:
df = df.filter(
    col("order_id").isNotNull() &
    col("order_item_id").isNotNull() &
    col("product_id").isNotNull() &
    col("seller_id").isNotNull()
)

### 8. Remove Duplicates 

In [0]:
# Check for duplicates before removing
print(f"Rows before deduplication: {df.count()}")
print(f"Distinct rows based on primary key: {df.select('order_id', 'order_item_id').distinct().count()}")

# Drop duplicates based on primary key columns
df = df.dropDuplicates(['order_id', 'order_item_id'])

print(f"Rows after deduplication: {df.count()}")

## Quality Checks

In [0]:
print(f"Total rows after cleaning: {df.count()}")
print(f"Unique (order_id, order_item_id): {df.select('order_id', 'order_item_id').distinct().count()}")
print(f"Distinct orders: {df.select('order_id').distinct().count()}")
print(f"Distinct products: {df.select('product_id').distinct().count()}")
print(f"Distinct sellers: {df.select('seller_id').distinct().count()}")
print(f"Rows with invalid amounts: {df.filter(~col('has_valid_amounts')).count()}")
print(f"Null price: {df.filter(col('price').isNull()).count()}")
print(f"Null freight_value: {df.filter(col('freight_value').isNull()).count()}")
df.display()

## Write to silver layer

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("olist.silver.order_items")